# ==========================================================
# LIMPIEZA Y CONSTRUCCIÓN DE BASE - SABER 11
# ==========================================================

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob

# ----------------------------------------------------------
# 1. DEFINIR RUTAS DEL PROYECTO
# ----------------------------------------------------------

In [2]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "cleaned"
ANALYSIS_DIR = BASE_DIR / "data" / "analysis"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio base:", BASE_DIR)
print("Directorio raw:", RAW_DIR)

Directorio base: c:\Users\windows\Documents\Esteban 2025-1\Maestria\Proyecto
Directorio raw: c:\Users\windows\Documents\Esteban 2025-1\Maestria\Proyecto\data\raw


# ----------------------------------------------------------
# 2. LISTAR ARCHIVOS SABER 11
# ----------------------------------------------------------

In [3]:
archivos = sorted(glob.glob(str(RAW_DIR / "*.txt")))

print("Archivos encontrados:", len(archivos))

for f in archivos:
    print(Path(f).name)

Archivos encontrados: 18
Examen_Saber_11_20161.txt
Examen_Saber_11_20162.txt
Examen_Saber_11_20171.txt
Examen_Saber_11_20172.txt
Examen_Saber_11_20181.txt
Examen_Saber_11_20182.txt
Examen_Saber_11_20191.txt
Examen_Saber_11_20192.txt
Examen_Saber_11_20201.txt
Examen_Saber_11_20202.txt
Examen_Saber_11_20211.txt
Examen_Saber_11_20212.txt
Examen_Saber_11_20221.txt
Examen_Saber_11_20222.txt
Examen_Saber_11_20231.txt
Examen_Saber_11_20232.txt
Examen_Saber_11_20241.txt
Examen_Saber_11_20242.txt



# ----------------------------------------------------------
# 3. FUNCIÓN PARA CARGAR Y ESTANDARIZAR CADA ARCHIVO
# ----------------------------------------------------------

In [ ]:
def cargar_y_estandarizar_archivo(path):
    """Carga un archivo Saber 11 (sep=';', encoding='latin-1') y normaliza nombres de columnas."""
    df = pd.read_csv(path, sep=";", encoding="latin-1", low_memory=False, dtype=str)

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("á", "a", regex=False)
        .str.replace("é", "e", regex=False)
        .str.replace("í", "i", regex=False)
        .str.replace("ó", "o", regex=False)
        .str.replace("ú", "u", regex=False)
        .str.replace("ñ", "n", regex=False)
    )

    # Extraer periodo del nombre del archivo (ej. '20161' → anio=2016, semestre=1)
    stem = Path(path).stem  # 'Examen_Saber_11_20161'
    periodo = stem.split("_")[-1]   # '20161'
    df["periodo"]  = periodo
    df["anio"]     = int(periodo[:4])
    df["semestre"] = int(periodo[4])

    return df


# ----------------------------------------------------------
# 4. CARGAR TODAS LAS BASES
# ----------------------------------------------------------


In [ ]:
lista_dfs = []

for archivo in archivos:
    df_temp = cargar_y_estandarizar_archivo(archivo)
    lista_dfs.append(df_temp)
    print(f"  {Path(archivo).name}: {len(df_temp):,} filas x {df_temp.shape[1]} cols")

print(f"\nTotal archivos cargados: {len(lista_dfs)}")


# ----------------------------------------------------------
# 5. UNIR TODAS LAS BASES
# ----------------------------------------------------------

In [ ]:
df_saber11 = pd.concat(lista_dfs, ignore_index=True, sort=False)

print(f"Base unificada: {df_saber11.shape[0]:,} filas x {df_saber11.shape[1]} columnas")
print("\nColumnas disponibles:")
print(sorted(df_saber11.columns.tolist()))


# ----------------------------------------------------------
# 6. LIMPIEZA BÁSICA
# ----------------------------------------------------------

In [ ]:
# Eliminar duplicados exactos
n_antes = len(df_saber11)
df_saber11 = df_saber11.drop_duplicates()
print(f"Duplicados eliminados: {n_antes - len(df_saber11):,}")
print(f"Base después de dedup: {df_saber11.shape}")


# ----------------------------------------------------------
# 7. SELECCIONAR VARIABLES CLAVE
# ----------------------------------------------------------

In [ ]:
# Variables a conservar (comunes a todos los periodos)
COLS_COMUNES = [
    # Identificación y periodo
    "periodo", "anio", "semestre", "estu_consecutivo",
    # Geografía del colegio
    "cole_cod_depto_ubicacion", "cole_cod_mcpio_ubicacion",
    "cole_depto_ubicacion", "cole_mcpio_ubicacion",
    "cole_area_ubicacion", "cole_calendario", "cole_caracter",
    "cole_naturaleza", "cole_jornada", "cole_genero",
    # Estudiante
    "estu_genero", "estu_fechanacimiento", "estu_grado",
    "estu_inse_individual", "estu_nse_individual", "estu_nse_establecimiento",
    "estu_discapacidad", "estu_tieneetnia", "estu_etnia", "estu_nacionalidad",
    # Familia / socioeconómico
    "fami_estratovivienda", "fami_educacionmadre", "fami_educacionpadre",
    "fami_cuartoshogar", "fami_personashogar", "fami_numlibros",
    "fami_tieneautomovil", "fami_tienecomputador", "fami_tieneinternet",
    "fami_tienelavadora", "fami_tieneserviciotv",
    # Puntajes
    "punt_matematicas", "punt_lectura_critica", "punt_c_naturales",
    "punt_ingles", "punt_sociales_ciudadanas", "punt_global",
    # Percentiles y desempeño
    "percentil_global", "percentil_matematicas", "percentil_lectura_critica",
    "desemp_matematicas", "desemp_lectura_critica",
]

# Variables solo presentes en 2022+ (se rellenan con NaN en periodos anteriores)
COLS_2022_PLUS = [
    "estu_horassemanatrabaja", "fami_situacioneconomica",
    "fami_tienemotocicleta", "fami_trabajolabormadre", "fami_trabajolaborpadre",
]

TODAS_LAS_COLS = COLS_COMUNES + COLS_2022_PLUS

# Seleccionar columnas existentes; agregar NaN para las que falten
cols_presentes = [c for c in TODAS_LAS_COLS if c in df_saber11.columns]
cols_faltantes = [c for c in TODAS_LAS_COLS if c not in df_saber11.columns]

base_limpia = df_saber11[cols_presentes].copy()
for c in cols_faltantes:
    base_limpia[c] = np.nan

base_limpia = base_limpia[TODAS_LAS_COLS]  # reordenar

print(f"Variables seleccionadas: {base_limpia.shape[1]}")
print(f"Columnas faltantes (se añadieron como NaN): {cols_faltantes}")


# ----------------------------------------------------------
# 8. CONVERTIR PUNTAJES A NUMÉRICO
# ----------------------------------------------------------

In [ ]:
# ----------------------------------------------------------
# 8A. Puntajes, percentiles, códigos geográficos → numérico
# ----------------------------------------------------------
COLS_NUM = [
    "punt_matematicas", "punt_lectura_critica", "punt_c_naturales",
    "punt_ingles", "punt_sociales_ciudadanas", "punt_global",
    "percentil_global", "percentil_matematicas", "percentil_lectura_critica",
    "estu_inse_individual",
    "cole_cod_depto_ubicacion", "cole_cod_mcpio_ubicacion",
]
for col in COLS_NUM:
    base_limpia[col] = pd.to_numeric(base_limpia[col], errors="coerce")

# ----------------------------------------------------------
# 8B. Fecha de nacimiento → edad aproximada
# ----------------------------------------------------------
base_limpia["estu_fechanacimiento"] = pd.to_datetime(
    base_limpia["estu_fechanacimiento"], format="%d/%m/%Y", errors="coerce"
)
base_limpia["edad_aprox"] = base_limpia["anio"] - base_limpia["estu_fechanacimiento"].dt.year
mask_edad = (base_limpia["edad_aprox"] < 13) | (base_limpia["edad_aprox"] > 25)
base_limpia.loc[mask_edad, "edad_aprox"] = np.nan

# ----------------------------------------------------------
# 8C. Estrato → numérico (extrae el dígito de 'Estrato 2')
# ----------------------------------------------------------
base_limpia["estrato"] = (
    base_limpia["fami_estratovivienda"].astype(str)
    .str.extract(r"(\d)")[0]
    .astype(float)
)

# ----------------------------------------------------------
# 8D. Variables binarias Si/No → 1/0 (Int8 soporta NaN)
# ----------------------------------------------------------
MAPA_SINO = {"Si": 1, "SI": 1, "si": 1, "S": 1,
             "No": 0, "NO": 0, "no": 0, "N": 0}
COLS_SINO = [
    "fami_tieneautomovil", "fami_tienecomputador", "fami_tieneinternet",
    "fami_tienelavadora", "fami_tieneserviciotv", "fami_tienemotocicleta",
]
for col in COLS_SINO:
    base_limpia[col] = base_limpia[col].map(MAPA_SINO).astype("Int8")

# ----------------------------------------------------------
# 8E. Texto → strip + upper; reemplazar NAN/NONE por NaN real
# ----------------------------------------------------------
COLS_TEXTO = [
    "cole_area_ubicacion", "cole_calendario", "cole_caracter",
    "cole_naturaleza", "cole_jornada", "cole_genero",
    "cole_depto_ubicacion", "cole_mcpio_ubicacion",
    "estu_genero", "estu_nacionalidad", "estu_discapacidad", "estu_tieneetnia",
    "fami_educacionmadre", "fami_educacionpadre",
    "fami_cuartoshogar", "fami_personashogar", "fami_numlibros",
    "estu_nse_individual", "estu_nse_establecimiento",
    "desemp_matematicas", "desemp_lectura_critica",
    "estu_horassemanatrabaja", "fami_situacioneconomica",
    "fami_trabajolabormadre", "fami_trabajolaborpadre",
]
for col in COLS_TEXTO:
    if col in base_limpia.columns:
        base_limpia[col] = (
            base_limpia[col].astype(str).str.strip().str.upper()
            .replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
        )

# ----------------------------------------------------------
# 8F. Filtrar solo grado 11 con puntaje global válido
# ----------------------------------------------------------
n_antes = len(base_limpia)
base_limpia["estu_grado"] = base_limpia["estu_grado"].astype(str).str.strip()
base_limpia = base_limpia[base_limpia["estu_grado"] == "11"].copy()
base_limpia = base_limpia[base_limpia["punt_global"].notna()].copy()

print(f"Filas antes del filtro: {n_antes:,}")
print(f"Filas después (grado 11 + puntaje válido): {len(base_limpia):,}")
print(f"Registros eliminados: {n_antes - len(base_limpia):,}")

# Diagnóstico rápido de faltantes en puntajes
miss = (base_limpia[["punt_global","punt_matematicas","punt_lectura_critica"]].isna().mean()*100).round(2)
print("\n% faltantes en puntajes clave:")
print(miss)


# ----------------------------------------------------------
# 9. GUARDAR BASE COMPLETA
# ----------------------------------------------------------

In [ ]:
import os

# --- Parquet (formato principal, eficiente para análisis) ---
ruta_parquet = CLEAN_DIR / "saber11_individual_limpio.parquet"
base_limpia.reset_index(drop=True).to_parquet(ruta_parquet, index=False)
print(f"Parquet guardado en: {ruta_parquet}")
print(f"  Tamaño: {os.path.getsize(ruta_parquet) / 1e6:.1f} MB")

# Verificación rápida
df_check = pd.read_parquet(ruta_parquet)
print(f"\nVerificación: {df_check.shape[0]:,} filas x {df_check.shape[1]} columnas")
print("Columnas:", df_check.columns.tolist())

# Estadísticas descriptivas de puntajes
print("\nEstadísticas de puntajes:")
print(df_check[["punt_global","punt_matematicas","punt_lectura_critica","punt_c_naturales","punt_ingles","punt_sociales_ciudadanas"]].describe().round(2))
